# Tidal Analysis by Sensor

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import folium

os.chdir(r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\2_tidal_analysis")

In [ ]:
# read in the cleaned final_deployed_sensors.geojson from 2_FloodNet_Sensor_Locations.ipynb
floodnet_sensor_mapping = gpd.read_file('../final_deployed_sensors.geojson')

In [ ]:
# load floodplain boundaries (same source as 2_FloodNet_Sensor_Locations.ipynb)
from shapely.geometry import shape

floodplains_df = pd.read_json('https://data.cityofnewyork.us/resource/ek8y-fsqz.json')
floodplains_df['the_geom'] = floodplains_df['the_geom'].apply(shape)
geo_floodplains = gpd.GeoDataFrame(floodplains_df, geometry='the_geom', crs='EPSG:4326')

In [ ]:
# Final map: same as m3 in 2_FloodNet_Sensor_Locations.ipynb
final_sensors_to_show = floodnet_sensor_mapping[
    floodnet_sensor_mapping['in_floodplain'] |
    (floodnet_sensor_mapping['deploy_type'] == 'coastal') |
    (floodnet_sensor_mapping['tidally_influenced'] == 'Yes')
].copy()

final_sensors_to_show['deploy_type'] = final_sensors_to_show['deploy_type'].fillna('unknown')

def get_color(row):
    is_coastal = row['deploy_type'] == 'coastal'
    is_tidal = row['tidally_influenced'] == 'Yes'
    if is_coastal and is_tidal:
        return 'purple'
    elif is_coastal:
        return 'blue'
    elif is_tidal:
        return 'green'
    elif row['deploy_type'] == 'unknown':
        return 'orange'
    else:
        return 'gray'

legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background: white; padding: 6px 10px; border: 1px solid gray;
            border-radius: 5px; font-size: 9px; line-height: 1.5;">
  <b>Sensor Type</b><br>
  <span style="color:purple;">●</span> Coastal + Tidally Influenced<br>
  <span style="color:blue;">●</span> Coastal<br>
  <span style="color:green;">●</span> Tidally Influenced<br>
  <span style="color:orange;">●</span> In floodplain (deploy type unknown)<br>
  <span style="color:gray;">●</span> In floodplain only
</div>
"""

m3 = folium.Map(location=[final_sensors_to_show.geometry.y.mean(), final_sensors_to_show.geometry.x.mean()], zoom_start=11)

folium.GeoJson(geo_floodplains, style_function=lambda x: {'color': 'cadetblue', 'fillOpacity': 0.3}).add_to(m3)

for _, row in final_sensors_to_show.iterrows():
    color = get_color(row)
    folium.CircleMarker(
        location=[row['geometry'].y, row['geometry'].x],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        tooltip=f"{row['sensor_name']} | deploy: {row['deploy_type']} | tidal: {row['tidally_influenced']}"
    ).add_to(m3)

m3.get_root().html.add_child(folium.Element(legend_html))
m3

## Bringing in sensor data

In [5]:
# reading in queried data from FloodNet Sensors from NYC Open Data (flood events after 2025-01-01)
flood_data = pd.read_json('https://data.cityofnewyork.us/resource/aq7i-eu5q.json?$query=SELECT%0A%20%20%60sensor_name%60%2C%0A%20%20%60sensor_id%60%2C%0A%20%20%60flood_start_time%60%2C%0A%20%20%60flood_end_time%60%2C%0A%20%20%60max_depth_inches%60%2C%0A%20%20%60flood_profile_depth_inches%60%2C%0A%20%20%60flood_profile_time_secs%60%2C%0A%20%20%60duration_mins%60%0AWHERE%20%60flood_start_time%60%20%3E%20%222025-01-01T00%3A00%3A00%22%20%3A%3A%20floating_timestamp%0ALIMIT%2050000')

In [6]:
display(flood_data.head())

,sensor_name,sensor_id,flood_start_time,flood_end_time,max_depth_inches,flood_profile_depth_inches,flood_profile_time_secs,duration_mins
0,BX - Ditmars St/Hunter Ave 2,BX-ditmars-st-hunter-ave-1kwrk0,2025-01-01 03:04:54,2025-01-01 04:06:00,1.46,"[0.00, 1.02, 1.46, 1.42, 1.46, 1.46, 1.42, 1.4...","[0, 190, 379, 443, 506, 569, 633, 759, 885, 94...",61.09
1,Q - Beach 84 St,Q-beach-84-st-0me680,2025-01-01 12:07:45,2025-01-01 15:19:58,2.64,"[0.00, 0.43, 0.55, 0.71, 0.98, 1.14, 1.18, 1.1...","[0, 695, 946, 1263, 2018, 2773, 2900, 3215, 45...",192.22
2,Q - Brookville Blvd/ Snake Rd 1,Q-brookville-blvd-rockaway-blvd-1-23ndo0,2025-01-01 13:16:13,2025-01-01 14:08:11,1.06,"[0.00, 0.47, 0.51, 0.59, 0.59, 0.59, 0.71, 0.7...","[0, 63, 127, 190, 254, 317, 381, 444, 508, 571...",51.98
3,BX - Ditmars St/Hunter Ave 2,BX-ditmars-st-hunter-ave-1kwrk0,2025-01-01 15:57:50,2025-01-01 17:48:28,4.76,"[0.00, 0.98, 1.34, 2.24, 2.80, 2.91, 3.19, 3.2...","[0, 443, 633, 885, 1074, 1200, 1263, 1326, 145...",110.63
4,Q - Beach 84 St,Q-beach-84-st-0me680,2025-01-11 09:28:04,2025-01-11 11:28:57,1.34,"[0.00, 0.55, 0.63, 0.63, 0.71, 0.71, 0.83, 0.8...","[0, 314, 631, 818, 881, 1012, 1326, 1517, 1890...",120.89


### Spatial join: flood events at floodplain sensors

Join `flood_data` (flood events by sensor name) to `final_sensors_to_show` (floodplain-relevant sensors with geometry). Inner join on normalized sensor name — keeps only floods at sensors already filtered to floodplains.

In [7]:
import re

def normalize_name(name):
    return re.sub(r'\s+', ' ', str(name).strip().lower())

# Normalize sensor names so flood_data and final_sensors_to_show can be joined despite spacing/case differences
flood_data['_key'] = flood_data['sensor_name'].apply(normalize_name)

# Pull only the columns needed for the join, plus geometry/classification flags
sensor_lookup = final_sensors_to_show[
    ['sensor_name', 'geometry', 'deploy_type', 'tidally_influenced', 'in_floodplain']
].copy()
# Build the same normalized key on the sensor side for matching
sensor_lookup['_key'] = sensor_lookup['sensor_name'].apply(normalize_name)

# Inner join keeps only flood events whose sensor matches a known floodplain sensor
flood_geo = flood_data.merge(
    sensor_lookup.drop(columns='sensor_name'),
    on='_key',
    how='inner'
).drop(columns='_key')

# Restore geometry/CRS since merge returns a plain DataFrame
flood_geo = gpd.GeoDataFrame(flood_geo, geometry='geometry', crs='EPSG:4326')

# Sanity check: how many flood events survived the join, and how many distinct sensors they cover
print(f"Total flood events in floodplain sensors: {len(flood_geo)}")
print(f"Unique sensors matched: {flood_geo['sensor_name'].nunique()} / {final_sensors_to_show['sensor_name'].nunique()} in floodplain set")

# Identify flood_data sensor names that failed to match any floodplain sensor (naming mismatches, retired sensors, etc.)
unmatched = set(flood_data['sensor_name'].str.strip().str.lower()) - set(sensor_lookup['_key'])
# Report count of unmatched sensor names for follow-up investigation
print(f"Flood-data sensors with no floodplain match: {len(unmatched)}")

# Preview the joined dataset
flood_geo[['sensor_name', 'flood_start_time', 'flood_end_time', 'max_depth_inches',
           'duration_mins', 'deploy_type', 'tidally_influenced', 'in_floodplain', 'geometry']].head()

Total flood events in floodplain sensors: 973
Unique sensors matched: 127 / 200 in floodplain set
Flood-data sensors with no floodplain match: 122


,sensor_name,flood_start_time,flood_end_time,max_depth_inches,duration_mins,deploy_type,tidally_influenced,in_floodplain,geometry
0,BX - Ditmars St/Hunter Ave 2,2025-01-01 03:04:54,2025-01-01 04:06:00,1.46,61.09,coastal,Yes,True,POINT (-73.78953 40.8493)
1,Q - Beach 84 St,2025-01-01 12:07:45,2025-01-01 15:19:58,2.64,192.22,coastal,Yes,True,POINT (-73.80996 40.59136)
2,Q - Brookville Blvd/ Snake Rd 1,2025-01-01 13:16:13,2025-01-01 14:08:11,1.06,51.98,coastal,Yes,True,POINT (-73.74448 40.6439)
3,BX - Ditmars St/Hunter Ave 2,2025-01-01 15:57:50,2025-01-01 17:48:28,4.76,110.63,coastal,Yes,True,POINT (-73.78953 40.8493)
4,Q - Beach 84 St,2025-01-11 09:28:04,2025-01-11 11:28:57,1.34,120.89,coastal,Yes,True,POINT (-73.80996 40.59136)


### Assign each flood sensor to its nearest tidal station

In [8]:
# bring in tidal data
tidal_unified = gpd.read_file('tidal_unified.geojson')

In [9]:
# One point per tidal station (tidal_unified is a time series)
# Collapse the tidal time series down to one row per station for spatial join
tidal_stations = (
    tidal_unified[['station_id', 'station_name', 'geometry']]
    .drop_duplicates('station_id')
    .reset_index(drop=True)
)

# Project to UTM 18N (meters) for accurate distance calculation over NYC
# One row per sensor; reproject to UTM 18N since sjoin_nearest measures distance in the CRS's units (degrees are not uniform distance)
sensor_pts = (
    flood_geo[['sensor_name', 'geometry']]
    .drop_duplicates('sensor_name')
    .to_crs('EPSG:32618')
)
# Match sensor_pts' CRS so distances are computed correctly
stations_proj = tidal_stations.to_crs('EPSG:32618')

# For each sensor, find its closest tidal station and the distance to it (in meters)
nearest = gpd.sjoin_nearest(
    sensor_pts,
    stations_proj,
    how='left',
    distance_col='dist_to_station_m'
)[['sensor_name', 'station_id', 'station_name', 'dist_to_station_m']]

# Attach the nearest-station assignment back to every flood event for that sensor
flood_geo = flood_geo.merge(nearest, on='sensor_name', how='left')

# Sanity check: confirm every flood event got a station assignment
print(f"Flood events assigned to a tidal station: {flood_geo['station_id'].notna().sum()} / {len(flood_geo)}")
# Preview one row per sensor, sorted by distance to its assigned station
flood_geo[['sensor_name', 'station_id', 'station_name', 'dist_to_station_m']].drop_duplicates('sensor_name').sort_values('dist_to_station_m')

Flood events assigned to a tidal station: 973 / 973


,sensor_name,station_id,station_name,dist_to_station_m
603,M - Broad St/South St,8518750,The Battery,286.202415
654,M - Peck Slip/Front St,8518750,The Battery,1327.927919
864,Q - Norton Dr/ Westbourne Ave,01311850,JAMAICA BAY AT INWOOD NY,1339.392499
604,M - Barclay St/West St,8518750,The Battery,1543.599352
967,M - Market Slip/South St,8518750,The Battery,1971.053139
...,...,...,...,...
643,BX - Lincoln Ave / Bruckner Blvd,8518750,The Battery,13748.774887
11,M - W 125th St/12th Ave,8518750,The Battery,13824.978584
463,M - 10th Ave/ W 211th St,8516945,Kings Point,14306.487954
459,M - Nagle Ave/Dyckman St,8516945,Kings Point,14656.669529


### Temporal join: lag to nearest high tide

For each flood event, find the nearest high-tide reading (`tidal_phase == 'H'`) at its assigned station. The **signed lag** — `flood_start_time − nearest_high_tide_time` — preserves direction:

- **Negative**: flood begins before high tide (rising tide)
- **Positive**: flood begins after high tide (falling tide)
- **Near ±6.21 h**: flood is near low tide (equidistant from two high tides)

A cluster near zero signals tidal control at high tide; a cluster near ±6.21 h signals flooding near low tide.

In [10]:
# Filter tidal readings down to just high-tide points, one lookup table per station
high_tides = (
    tidal_unified[tidal_unified['tidal_phase'] == 'H']
    [['station_id', 'datetime', 'water_level_ft']]
    .copy()
    .sort_values('datetime')
)

# Ensure both timestamps are timezone-aware so subtraction below is valid
flood_geo['flood_start_time'] = pd.to_datetime(flood_geo['flood_start_time'], utc=True)

lag_hours_map = {}
high_tide_level_map = {}

# Process one tidal station at a time so each flood event only searches its assigned station's high tides
for station_id, flood_grp in flood_geo.groupby('station_id'):
    tides = high_tides[high_tides['station_id'] == station_id].reset_index(drop=True)
    if tides.empty:
        continue
    for idx, row in flood_grp.iterrows():
        t = row['flood_start_time']
        # Find the closest high tide in time, regardless of direction
        abs_diffs = (tides['datetime'] - t).abs() / pd.Timedelta(hours=1)
        best = abs_diffs.idxmin()
        # Signed lag preserves whether the flood was before (-) or after (+) that high tide
        signed_lag = (t - tides.loc[best, 'datetime']) / pd.Timedelta(hours=1)
        lag_hours_map[idx] = round(signed_lag, 3)
        high_tide_level_map[idx] = tides.loc[best, 'water_level_ft']

# Map results back onto flood_geo by its original index
flood_geo['lag_hours'] = pd.Series(lag_hours_map)
flood_geo['high_tide_level_ft'] = pd.Series(high_tide_level_map)

# Sanity check: confirm every flood event got a lag value
print(f"Lag computed for {flood_geo['lag_hours'].notna().sum()} / {len(flood_geo)} flood events")
flood_geo[['sensor_name', 'flood_start_time', 'lag_hours', 'high_tide_level_ft']].head(10)

Lag computed for 973 / 973 flood events


,sensor_name,flood_start_time,lag_hours,high_tide_level_ft
0,BX - Ditmars St/Hunter Ave 2,2025-01-01 03:04:54+00:00,-13.568,3.793
1,Q - Beach 84 St,2025-01-01 12:07:45+00:00,-1.671,3.730
2,Q - Brookville Blvd/ Snake Rd 1,2025-01-01 13:16:13+00:00,-0.530,3.730
3,BX - Ditmars St/Hunter Ave 2,2025-01-01 15:57:50+00:00,-0.686,3.793
4,Q - Beach 84 St,2025-01-11 09:28:04+00:00,-1.032,3.830
5,BX - Ditmars St/Hunter Ave 2,2025-01-11 11:00:57+00:00,-3.118,3.684
6,Q - Beach 84 St,2025-01-12 10:43:47+00:00,-0.770,3.870
7,Q - Beach 84 St,2025-01-13 11:17:00+00:00,-1.117,4.090
8,Q - Brookville Blvd/ Snake Rd 1,2025-01-13 12:15:51+00:00,-0.136,4.090
9,BX - Ditmars St/Hunter Ave 2,2025-02-03 07:16:51+00:00,0.048,3.849


In [11]:
print(flood_geo[['lag_hours', 'high_tide_level_ft']].describe())
print(flood_geo[['lag_hours', 'high_tide_level_ft']].nunique())

        lag_hours  high_tide_level_ft
count  973.000000          973.000000
mean    -0.626401            3.612778
std      2.252061            0.950563
min    -13.568000            1.090000
25%     -1.432000            2.992000
50%     -0.569000            3.870000
75%      0.061000            4.290000
max      6.113000            5.395000
lag_hours             881
high_tide_level_ft    313
dtype: int64


In [12]:
print(flood_geo['lag_hours'].isna().sum())
print(flood_geo['high_tide_level_ft'].isna().sum())

0
0


In [13]:
flood_geo[flood_geo['lag_hours'].isna()][['sensor_name', 'station_id', 'flood_start_time']].head(10)

,sensor_name,station_id,flood_start_time


### Statistical test: Rayleigh test for circular uniformity

Each signed lag is converted to a phase angle on the full tidal cycle:

**θ = (lag_hours / 12.42) × 2π**

- **θ = 0**: flooding exactly at high tide (lag = 0)
- **θ = ±π**: flooding at low tide (lag = ±6.21 h, both sides of the circle meet here)

The **Rayleigh test** asks whether flood phases are uniformly scattered (null) or cluster in a preferred direction. **R** (mean resultant length, 0–1) measures concentration.

**Classification** (requires `p < 0.05` and `R > 0.30`):

| Circular mean lag | Label |
|---|---|
| `abs(mean_lag) < 3.1 h` (within quarter-period of high tide) | Yes – high tide |
| `abs(mean_lag) ≥ 3.1 h` (closer to low tide) | Yes – low tide |

Sensors with fewer than 3 flood events are flagged as insufficient data.

**Tide direction** (only set for `Yes` classifications): the sign of the circular mean lag holds regardless of whether the sensor clusters near high or low tide — negative means floods fall before the nearest high tide (rising limb), positive means after it (falling limb).


In [14]:
TIDAL_PERIOD_HRS = 12.42  # mean M2 semidiurnal tidal period
MIN_EVENTS = 3  # minimum flood events required to run the test on a sensor

def rayleigh_test(lags):
    """
    Rayleigh test for circular uniformity (Mardia & Jupp 2000).
    Lags are signed hours relative to nearest high tide:
      negative = before high tide, positive = after.
    Returns R, p-value, and circular mean lag (hours).
    """
    n = len(lags)
    if n < MIN_EVENTS:
        return np.nan, np.nan, np.nan
    lags = np.asarray(lags)
    # Map each lag onto the tidal cycle as an angle (0 = high tide, ±π = low tide)
    theta = (lags / TIDAL_PERIOD_HRS) * 2 * np.pi
    C = np.mean(np.cos(theta))
    S = np.mean(np.sin(theta))
    # R = mean resultant length: 0 = uniformly scattered, 1 = all lags identical
    R = np.sqrt(C**2 + S**2)
    mean_angle = np.arctan2(S, C)
    # Convert the mean angle back to hours for interpretability
    mean_lag = (mean_angle / (2 * np.pi)) * TIDAL_PERIOD_HRS
    z = n * R**2
    # Approximate p-value for the Rayleigh test statistic z
    p = np.exp(-z) * (1 + (2*z - z**2)/(4*n) - (24*z - 132*z**2 + 76*z**3 - 9*z**4)/(288*n**2))
    return round(R, 3), round(float(np.clip(p, 0, 1)), 4), round(mean_lag, 2)


# Only rows with a computed lag can be tested
flood_valid = flood_geo.dropna(subset=['lag_hours'])
rows = []

# Run the Rayleigh test per sensor and classify its tidal relationship
for sensor, grp in flood_valid.groupby('sensor_name'):
    lags = grp['lag_hours'].values
    n = len(lags)
    R, p, mean_lag = rayleigh_test(lags)

    if n < MIN_EVENTS:
        tidal_class = 'Insufficient data'
        tide_direction = np.nan
    elif p < 0.05 and R > 0.30:
        # Significant clustering: label by whether it's nearer high tide or low tide
        if abs(mean_lag) < TIDAL_PERIOD_HRS / 4:  # quarter-period boundary = 3.1 h
            tidal_class = 'Yes – high tide'
        else:
            tidal_class = 'Yes – low tide'
        # Sign holds regardless of high/low: negative = before the nearest high tide
        # (rising limb), positive = after it (falling limb)
        tide_direction = 'rising' if mean_lag < 0 else 'falling'
    else:
        tidal_class = 'No'
        tide_direction = np.nan

    rows.append({
        'sensor_name': sensor,
        'n_floods': n,
        'mean_lag_hrs': round(np.mean(lags), 2),
        'circular_mean_lag_hrs': mean_lag,
        'R': R,
        'p_value': p if not np.isnan(p) else np.nan,
        'tidal_class_data': tidal_class,
        'tide_direction': tide_direction,
        'tidal_class_existing': grp['tidally_influenced'].iloc[0]
    })

# One row per sensor, most statistically significant first
sensor_class = (
    pd.DataFrame(rows)
    .sort_values('p_value', na_position='last')
    .reset_index(drop=True)
)
sensor_class


,sensor_name,n_floods,mean_lag_hrs,circular_mean_lag_hrs,R,p_value,tidal_class_data,tide_direction,tidal_class_existing
0,BX - Ditmars St/Hunter Ave 2,142,-1.17,-1.30,0.758,0.0,Yes – high tide,rising,Yes
1,Q - Brookville Blvd/ Snake Rd 1,92,-0.25,-0.24,0.978,0.0,Yes – high tide,rising,Yes
2,Q - Brookville Blvd/ Snake Rd 2,56,-0.29,-0.39,0.860,0.0,Yes – high tide,rising,Yes
3,Q - Brookville Blvd/ Snake Rd 3,61,-0.23,-0.09,0.714,0.0,Yes – high tide,rising,Yes
4,Q - Beach 84 St,82,-1.03,-1.05,0.872,0.0,Yes – high tide,rising,Yes
...,...,...,...,...,...,...,...,...,...
122,SI - Catherine Ct/Jewett Ave,1,2.08,NaN,NaN,NaN,Insufficient data,NaN,No
123,SI - Genesee Ave/Richmond Ave,2,0.87,NaN,NaN,NaN,Insufficient data,NaN,No
124,SI - Olympia Blvd/ Mapleton Ave,1,1.56,NaN,NaN,NaN,Insufficient data,NaN,Yes
125,SI - Rector St/Richmond Terr,2,-1.13,NaN,NaN,NaN,Insufficient data,NaN,Yes


In [15]:
print(sensor_class["tidal_class_data"].value_counts())

pd.crosstab(sensor_class['tidal_class_data'], sensor_class['tidal_class_existing'])


tidal_class_data
Insufficient data    65
No                   41
Yes – high tide      19
Yes – low tide        2
Name: count, dtype: int64


tidal_class_existing,No,Yes
tidal_class_data,,
Insufficient data,55,10
No,29,12
Yes – high tide,3,16
Yes – low tide,2,0


In [16]:
#viewing rows with mismatched tidal data and existing classification
data_says_yes = sensor_class['tidal_class_data'].str.startswith('Yes')
existing_says_yes = sensor_class['tidal_class_existing'] == 'Yes'

mismatches = sensor_class[data_says_yes != existing_says_yes].sort_values('n_floods')
display(mismatches)
print(mismatches.shape)

,sensor_name,n_floods,mean_lag_hrs,circular_mean_lag_hrs,R,p_value,tidal_class_data,tide_direction,tidal_class_existing
116,Q - Beach 43rd St,1,0.95,NaN,NaN,NaN,Insufficient data,NaN,Yes
126,SI - Seaview Ave/Hylan Blvd,1,-4.95,NaN,NaN,NaN,Insufficient data,NaN,Yes
124,SI - Olympia Blvd/ Mapleton Ave,1,1.56,NaN,NaN,NaN,Insufficient data,NaN,Yes
119,Q - Deerfield Rd/Beach 28th St,1,-2.33,NaN,NaN,NaN,Insufficient data,NaN,Yes
108,Q - 231st St/ 148th Ave,1,0.99,NaN,NaN,NaN,Insufficient data,NaN,Yes
113,Q - Beach 140th St/Cronston Ave,1,-5.17,NaN,NaN,NaN,Insufficient data,NaN,Yes
97,M - Indian Rd/W 215th St,1,-4.36,NaN,NaN,NaN,Insufficient data,NaN,Yes
115,Q - Beach 42nd St/Beach Channel Dr,2,-0.80,NaN,NaN,NaN,Insufficient data,NaN,Yes
117,Q - Beach 72nd St/Almeda Ave,2,2.07,NaN,NaN,NaN,Insufficient data,NaN,Yes
125,SI - Rector St/Richmond Terr,2,-1.13,NaN,NaN,NaN,Insufficient data,NaN,Yes


(27, 9)


## Checking values

In [17]:
false_negative_sensors = [
    "Q - Brookville Blvd/ 149th St",
    "SI - Hylan Blvd/ Jefferson Blvd",
    "SI - Bedford Ave/Kiswick St",
    "BX - Tier St/William Ave",
    "SI - Boundary Ave/Hamden Ave",
    "Q - Beach 49th St/Rockaway Beach Blvd",
    "SI - McLaughlin St/Agnes Pl",
    "Q - Beach Channel Dr/Beach 48th St",
    "SI - Minthorne St/ Victory Blvd",
    "SI - Baden Pl/ Mapleton Ave",
    "Q - Norton Dr/ Westbourne Ave",
    "SI - Grimsby St/ Mapleton Ave"
]

for sensor in false_negative_sensors:
    print(f"\n{'='*80}")
    print(sensor)
    print(f"{'='*80}")
    display(flood_geo[flood_geo['sensor_name'] == sensor])


Q - Brookville Blvd/ 149th St


,sensor_name,sensor_id,flood_start_time,flood_end_time,max_depth_inches,flood_profile_depth_inches,flood_profile_time_secs,duration_mins,geometry,deploy_type,tidally_influenced,in_floodplain,station_id,station_name,dist_to_station_m,lag_hours,high_tide_level_ft
478,Q - Brookville Blvd/ 149th St,Q-brookville-blvd-149th-ave-1zbc0d,2025-10-30 19:52:46+00:00,2025-10-30 20:47:46,2.76,"[0.00, 0.91, 1.30, 1.34, 2.05, 2.72, 2.76, 2.7...","[0, 180, 240, 420, 480, 600, 660, 720, 840, 90...",55.00,POINT (-73.74644 40.65265),coastal,Yes,True,01311850,JAMAICA BAY AT INWOOD NY,4018.627886,0.879,4.29
527,Q - Brookville Blvd/ 149th St,Q-brookville-blvd-149th-ave-1zbc0d,2025-12-17 23:00:16+00:00,2025-12-17 23:39:16,1.14,"[0.00, 0.59, 0.67, 0.67, 0.67, 0.79, 0.79, 0.8...","[0, 300, 360, 420, 480, 540, 600, 660, 720, 78...",39.00,POINT (-73.74644 40.65265),coastal,Yes,True,01311850,JAMAICA BAY AT INWOOD NY,4018.627886,-0.496,1.38
858,Q - Brookville Blvd/ 149th St,Q-brookville-blvd-149th-ave-1zbc0d,2026-05-20 23:28:17+00:00,2026-05-21 00:10:18,1.06,"[0.59, 0.59, 0.59, 0.59, 0.75, 0.91, 0.91, 0.9...","[0, 120, 180, 240, 300, 360, 420, 480, 540, 60...",42.01,POINT (-73.74644 40.65265),coastal,Yes,True,01311850,JAMAICA BAY AT INWOOD NY,4018.627886,-4.729,3.08



SI - Hylan Blvd/ Jefferson Blvd


,sensor_name,sensor_id,flood_start_time,flood_end_time,max_depth_inches,flood_profile_depth_inches,flood_profile_time_secs,duration_mins,geometry,deploy_type,tidally_influenced,in_floodplain,station_id,station_name,dist_to_station_m,lag_hours,high_tide_level_ft
179,SI - Hylan Blvd/ Jefferson Blvd,SI-hylan-blvd-jefferson-blvd-1nhfo0,2025-07-14 22:57:36+00:00,2025-07-14 23:13:36,3.82,"[0.00, 1.02, 1.97, 2.48, 3.50, 3.78, 3.82, 3.8...","[0, 120, 180, 240, 300, 360, 420, 480, 540, 60...",16.0,POINT (-74.09849 40.58106),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5383.616982,-4.440,2.99
225,SI - Hylan Blvd/ Jefferson Blvd,SI-hylan-blvd-jefferson-blvd-1nhfo0,2025-07-31 19:47:44+00:00,2025-07-31 20:07:44,5.67,"[0.00, 1.61, 2.13, 2.87, 4.33, 4.49, 4.96, 5.3...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",20.0,POINT (-74.09849 40.58106),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5383.616982,2.196,2.74
526,SI - Hylan Blvd/ Jefferson Blvd,SI-hylan-blvd-jefferson-blvd-1nhfo0,2025-12-17 22:48:51+00:00,2025-12-18 00:49:51,1.89,"[0.00, 0.00, 0.43, 0.47, 0.59, 0.47, 0.47, 0.0...","[0, 60, 120, 240, 300, 360, 420, 540, 780, 840...",121.0,POINT (-74.09849 40.58106),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5383.616982,-0.386,1.27



SI - Bedford Ave/Kiswick St


...

,sensor_name,sensor_id,flood_start_time,flood_end_time,max_depth_inches,flood_profile_depth_inches,flood_profile_time_secs,duration_mins,geometry,deploy_type,tidally_influenced,in_floodplain,station_id,station_name,dist_to_station_m,lag_hours,high_tide_level_ft
181,SI - Baden Pl/ Mapleton Ave,SI-baden-pl-mapleton-ave-1dejs0,2025-07-14 23:01:32+00:00,2025-07-15 01:20:22,1.77,"[0.00, 1.34, 1.30, 1.34, 1.54, 1.50, 1.54, 1.5...","[0, 253, 442, 568, 694, 757, 820, 883, 946, 10...",138.84,POINT (-74.09039 40.57319),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5068.014421,-4.374,2.99
226,SI - Baden Pl/ Mapleton Ave,SI-baden-pl-mapleton-ave-1dejs0,2025-07-31 19:52:30+00:00,2025-07-31 22:41:28,2.20,"[0.00, 0.59, 0.94, 1.57, 1.61, 1.65, 1.61, 1.5...","[0, 64, 126, 252, 504, 568, 756, 820, 1072, 11...",168.96,POINT (-74.09039 40.57319),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5068.014421,2.275,2.74
431,SI - Baden Pl/ Mapleton Ave,SI-baden-pl-mapleton-ave-1dejs0,2025-10-30 18:33:31+00:00,2025-10-30 21:18:44,3.66,"[0.00, 0.00, 0.87, 1.93, 2.01, 2.09, 2.36, 2.4...","[0, 63, 126, 253, 316, 379, 505, 568, 694, 757...",165.22,POINT (-74.09039 40.57319),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5068.014421,-0.241,4.48
595,SI - Baden Pl/ Mapleton Ave,SI-baden-pl-mapleton-ave-1dejs0,2026-03-06 04:44:42+00:00,2026-03-06 07:11:02,1.61,"[0.00, 0.47, 0.51, 0.51, 0.59, 0.67, 0.71, 0.7...","[0, 190, 253, 442, 506, 632, 759, 884, 1201, 1...",146.34,POINT (-74.09039 40.57319),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5068.014421,2.345,3.36
688,SI - Baden Pl/ Mapleton Ave,SI-baden-pl-mapleton-ave-1dejs0,2026-03-29 11:12:57+00:00,2026-03-29 11:27:40,0.55,"[0.00, 0.00, 0.51, 0.51, 0.55, 0.51, 0.51, 0.4...","[0, 63, 252, 315, 379, 442, 568, 631, 694, 883]",14.72,POINT (-74.09039 40.57319),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5068.014421,1.516,2.34
717,SI - Baden Pl/ Mapleton Ave,SI-baden-pl-mapleton-ave-1dejs0,2026-04-15 21:49:10+00:00,2026-04-15 22:11:15,1.06,"[0.00, 0.00, 0.59, 1.02, 1.06, 1.06, 0.98, 0.8...","[0, 63, 126, 251, 378, 441, 567, 630, 693, 756...",22.08,POINT (-74.09039 40.57319),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5068.014421,-1.281,3.24
808,SI - Baden Pl/ Mapleton Ave,SI-baden-pl-mapleton-ave-1dejs0,2026-05-18 02:19:55+00:00,2026-05-18 03:05:11,0.55,"[0.00, 0.43, 0.51, 0.51, 0.55, 0.55, 0.51, 0.5...","[0, 63, 190, 253, 316, 443, 569, 696, 760, 823...",45.26,POINT (-74.09039 40.57319),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5068.014421,1.232,3.75



Q - Norton Dr/ Westbourne Ave


,sensor_name,sensor_id,flood_start_time,flood_end_time,max_depth_inches,flood_profile_depth_inches,flood_profile_time_secs,duration_mins,geometry,deploy_type,tidally_influenced,in_floodplain,station_id,station_name,dist_to_station_m,lag_hours,high_tide_level_ft
864,Q - Norton Dr/ Westbourne Ave,Q-norton-dr-westbourne-ave-1-2wif1c,2026-05-23 21:06:08+00:00,2026-05-24 01:08:08,1.10,"[0.00, 0.43, 0.47, 0.47, 0.47, 0.47, 0.47, 0.4...","[0, 60, 120, 179, 239, 299, 359, 419, 479, 539...",242.00,POINT (-73.76873 40.60887),coastal,Yes,True,01311850,JAMAICA BAY AT INWOOD NY,1339.392499,2.202,2.64
868,Q - Norton Dr/ Westbourne Ave,Q-norton-dr-westbourne-ave-1-2wif1c,2026-05-24 07:24:09+00:00,2026-05-24 12:49:02,0.91,"[0.00, 0.00, 0.59, 0.59, 0.59, 0.55, 0.55, 0.5...","[0, 360, 2040, 2100, 2160, 2220, 2280, 2340, 2...",324.88,POINT (-73.76873 40.60887),coastal,Yes,True,01311850,JAMAICA BAY AT INWOOD NY,1339.392499,0.403,2.70
873,Q - Norton Dr/ Westbourne Ave,Q-norton-dr-westbourne-ave-1-2wif1c,2026-05-24 12:53:02+00:00,2026-05-24 19:38:01,1.54,"[0.00, 0.55, 0.55, 0.59, 0.59, 0.63, 0.67, 0.6...","[0, 720, 780, 840, 900, 959, 1019, 1079, 1139,...",404.99,POINT (-73.76873 40.60887),coastal,Yes,True,01311850,JAMAICA BAY AT INWOOD NY,1339.392499,5.884,2.70
880,Q - Norton Dr/ Westbourne Ave,Q-norton-dr-westbourne-ave-1-2wif1c,2026-05-25 12:14:59+00:00,2026-05-25 14:58:59,0.98,"[0.00, 0.63, 0.71, 0.75, 0.75, 0.75, 0.75, 0.7...","[0, 120, 180, 240, 300, 360, 420, 480, 540, 60...",164.00,POINT (-73.76873 40.60887),coastal,Yes,True,01311850,JAMAICA BAY AT INWOOD NY,1339.392499,4.250,2.55
937,Q - Norton Dr/ Westbourne Ave,Q-norton-dr-westbourne-ave-1-2wif1c,2026-06-16 00:24:12+00:00,2026-06-16 03:03:11,8.50,"[0.00, 0.43, 0.51, 0.71, 0.79, 0.94, 1.02, 1.0...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",159.00,POINT (-73.76873 40.60887),coastal,Yes,True,01311850,JAMAICA BAY AT INWOOD NY,1339.392499,-0.797,4.81
947,Q - Norton Dr/ Westbourne Ave,Q-norton-dr-westbourne-ave-1-2wif1c,2026-06-17 01:12:11+00:00,2026-06-17 03:15:09,4.13,"[0.00, 0.55, 0.63, 0.71, 0.71, 0.71, 0.71, 1.1...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",122.97,POINT (-73.76873 40.60887),coastal,Yes,True,01311850,JAMAICA BAY AT INWOOD NY,1339.392499,-0.897,4.52
956,Q - Norton Dr/ Westbourne Ave,Q-norton-dr-westbourne-ave-1-2wif1c,2026-06-18 02:08:07+00:00,2026-06-18 04:04:08,5.00,"[0.00, 0.63, 0.71, 0.75, 0.75, 0.75, 1.10, 1.1...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",116.01,POINT (-73.76873 40.60887),coastal,Yes,True,01311850,JAMAICA BAY AT INWOOD NY,1339.392499,-0.865,4.48



SI - Grimsby St/ Mapleton Ave


,sensor_name,sensor_id,flood_start_time,flood_end_time,max_depth_inches,flood_profile_depth_inches,flood_profile_time_secs,duration_mins,geometry,deploy_type,tidally_influenced,in_floodplain,station_id,station_name,dist_to_station_m,lag_hours,high_tide_level_ft
42,SI - Grimsby St/ Mapleton Ave,SI-grimsby-st-mapleton-ave-1de5w0,2025-03-21 03:04:00+00:00,2025-03-21 03:56:35,1.54,"[0.00, 0.47, 0.63, 0.71, 0.79, 0.94, 0.87, 0.9...","[0, 63, 126, 189, 251, 442, 505, 568, 631, 693...",52.59,POINT (-74.09345 40.57484),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5044.125998,-1.833,2.59
177,SI - Grimsby St/ Mapleton Ave,SI-grimsby-st-mapleton-ave-1de5w0,2025-07-14 22:56:49+00:00,2025-07-15 01:29:32,6.06,"[0.00, 1.14, 1.77, 2.95, 3.39, 3.94, 4.45, 5.0...","[0, 126, 189, 315, 379, 441, 631, 883, 1010, 1...",152.71,POINT (-74.09345 40.57484),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5044.125998,-4.453,2.99
224,SI - Grimsby St/ Mapleton Ave,SI-grimsby-st-mapleton-ave-1de5w0,2025-07-31 19:47:28+00:00,2025-07-31 22:51:41,6.54,"[0.00, 1.61, 1.77, 2.83, 3.54, 4.09, 4.57, 4.8...","[0, 127, 190, 254, 316, 379, 443, 506, 632, 69...",184.22,POINT (-74.09345 40.57484),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5044.125998,2.191,2.74
430,SI - Grimsby St/ Mapleton Ave,SI-grimsby-st-mapleton-ave-1de5w0,2025-10-30 18:33:20+00:00,2025-10-30 21:06:08,4.80,"[0.00, 0.43, 1.69, 2.64, 2.68, 3.03, 3.11, 3.4...","[0, 63, 127, 253, 316, 379, 505, 568, 694, 757...",152.80,POINT (-74.09345 40.57484),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5044.125998,-0.244,4.48
528,SI - Grimsby St/ Mapleton Ave,SI-grimsby-st-mapleton-ave-1de5w0,2025-12-18 20:24:58+00:00,2025-12-18 21:23:03,2.36,"[0.00, 0.83, 0.91, 1.34, 1.61, 1.81, 1.81, 1.8...","[0, 253, 444, 569, 759, 822, 885, 948, 1011, 1...",58.09,POINT (-74.09345 40.57484),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5044.125998,-3.484,2.03
537,SI - Grimsby St/ Mapleton Ave,SI-grimsby-st-mapleton-ave-1de5w0,2025-12-19 13:14:58+00:00,2025-12-19 14:45:31,1.77,"[0.00, 0.47, 0.71, 0.79, 0.83, 0.83, 0.87, 0.9...","[0, 126, 190, 252, 316, 379, 442, 506, 632, 69...",90.54,POINT (-74.09345 40.57484),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5044.125998,1.149,3.93
610,SI - Grimsby St/ Mapleton Ave,SI-grimsby-st-mapleton-ave-1de5w0,2026-03-12 03:52:33+00:00,2026-03-12 04:10:26,0.67,"[0.00, 0.00, 0.51, 0.51, 0.55, 0.55, 0.55, 0.6...","[0, 63, 190, 316, 379, 442, 505, 568, 632, 758...",17.89,POINT (-74.09345 40.57484),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5044.125998,-2.724,2.47
653,SI - Grimsby St/ Mapleton Ave,SI-grimsby-st-mapleton-ave-1de5w0,2026-03-12 11:40:27+00:00,2026-03-12 11:52:02,0.71,"[0.00, 0.51, 0.51, 0.71, 0.71, 0.59, 0.00]","[0, 63, 126, 189, 253, 442, 695]",11.59,POINT (-74.09345 40.57484),coastal,Yes,True,01376562,GREAT KILLS HARBOR AT GREAT KILLS NY,5044.125998,5.074,2.47


In [18]:
false_positive_sensors = [
    "BK - Brighton 6th St/Ocean View Ave",
    "Q - 101st St/160th Ave",
    "BX - Watson Ave/Close Ave",
    "SI - Snug Harbor Rd / Kissel Ave",
    "Q - Beach Chn Dr/ Beach 59th St"
]

for sensor in false_positive_sensors:
    print(f"\n{'='*80}")
    print(sensor)
    print(f"{'='*80}")
    display(flood_geo[flood_geo['sensor_name'] == sensor])


BK - Brighton 6th St/Ocean View Ave


,sensor_name,sensor_id,flood_start_time,flood_end_time,max_depth_inches,flood_profile_depth_inches,flood_profile_time_secs,duration_mins,geometry,deploy_type,tidally_influenced,in_floodplain,station_id,station_name,dist_to_station_m,lag_hours,high_tide_level_ft
827,BK - Brighton 6th St/Ocean View Ave,BK-brighton-6th-st-ocean-view-ave-2ub2dc,2026-05-20 22:31:48+00:00,2026-05-20 23:49:09,0.43,"[0.00, 0.43, 0.43, 0.43, 0.43, 0.00, 0.00, 0.0...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 600...",77.35,POINT (-73.96229 40.57963),pluvial,No,True,01311875,ROCKAWAY INLET NEAR FLOYD BENNETT FIELD NY,6527.888655,-5.370,3.04
840,BK - Brighton 6th St/Ocean View Ave,BK-brighton-6th-st-ocean-view-ave-2ub2dc,2026-05-20 22:53:48+00:00,2026-05-21 01:56:47,0.79,"[0.00, 0.00, 0.51, 0.51, 0.51, 0.51, 0.51, 0.5...","[0, 300, 360, 420, 480, 540, 600, 660, 960, 10...",182.98,POINT (-73.96229 40.57963),pluvial,No,True,01311875,ROCKAWAY INLET NEAR FLOYD BENNETT FIELD NY,6527.888655,-5.003,3.04
968,BK - Brighton 6th St/Ocean View Ave,BK-brighton-6th-st-ocean-view-ave-2ub2dc,2026-06-23 02:21:19+00:00,2026-06-23 04:09:18,1.54,"[0.00, 0.55, 0.59, 0.59, 0.67, 0.67, 0.71, 0.7...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",107.98,POINT (-73.96229 40.57963),pluvial,No,True,01311875,ROCKAWAY INLET NEAR FLOYD BENNETT FIELD NY,6527.888655,-4.945,2.39



Q - 101st St/160th Ave


,sensor_name,sensor_id,flood_start_time,flood_end_time,max_depth_inches,flood_profile_depth_inches,flood_profile_time_secs,duration_mins,geometry,deploy_type,tidally_influenced,in_floodplain,station_id,station_name,dist_to_station_m,lag_hours,high_tide_level_ft
724,Q - 101st St/160th Ave,Q-160th-ave-101st-st-2vfj80,2026-04-16 23:32:09+00:00,2026-04-17 00:40:09,2.32,"[0.00, 0.47, 0.55, 0.67, 0.83, 1.02, 1.06, 1.1...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",68.00,POINT (-73.83195 40.65829),pluvial,No,True,01311850,JAMAICA BAY AT INWOOD NY,7747.614737,-0.664,4.20
734,Q - 101st St/160th Ave,Q-160th-ave-101st-st-2vfj80,2026-04-18 00:15:08+00:00,2026-04-18 01:25:07,2.68,"[0.00, 0.55, 0.79, 0.94, 1.10, 1.14, 1.14, 1.2...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",69.98,POINT (-73.83195 40.65829),pluvial,No,True,01311850,JAMAICA BAY AT INWOOD NY,7747.614737,-0.748,4.26
907,Q - 101st St/160th Ave,Q-160th-ave-101st-st-2vfj80,2026-06-12 21:48:07+00:00,2026-06-12 22:32:07,1.34,"[0.00, 0.47, 0.47, 0.47, 0.55, 0.55, 0.55, 0.5...","[0, 60, 120, 180, 240, 299, 359, 419, 479, 539...",44.00,POINT (-73.83195 40.65829),pluvial,No,True,01311850,JAMAICA BAY AT INWOOD NY,7747.614737,-0.598,4.11



BX - Watson Ave/Close Ave


,sensor_name,sensor_id,flood_start_time,flood_end_time,max_depth_inches,flood_profile_depth_inches,flood_profile_time_secs,duration_mins,geometry,deploy_type,tidally_influenced,in_floodplain,station_id,station_name,dist_to_station_m,lag_hours,high_tide_level_ft
838,BX - Watson Ave/Close Ave,BX-watson-ave-close-ave-1zbbzc,2026-05-20 22:51:25+00:00,2026-05-20 23:18:25,2.32,"[0.00, 0.51, 1.50, 1.93, 1.93, 1.93, 1.93, 1.9...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",27.0,POINT (-73.88237 40.82563),pluvial,No,True,8516945,Kings Point,10052.444713,3.624,3.471
857,BX - Watson Ave/Close Ave,BX-watson-ave-close-ave-1zbbzc,2026-05-20 23:23:25+00:00,2026-05-20 23:37:25,2.17,"[0.00, 0.55, 1.65, 1.89, 1.97, 2.09, 2.09, 2.1...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",14.0,POINT (-73.88237 40.82563),pluvial,No,True,8516945,Kings Point,10052.444713,4.157,3.471
861,BX - Watson Ave/Close Ave,BX-watson-ave-close-ave-1zbbzc,2026-05-20 23:40:25+00:00,2026-05-20 23:46:25,9.65,"[0.00, 6.50, 9.65, 9.65, 9.65, 9.65, 0.00]","[0, 60, 120, 180, 240, 300, 360]",6.0,POINT (-73.88237 40.82563),pluvial,No,True,8516945,Kings Point,10052.444713,4.440,3.471



SI - Snug Harbor Rd / Kissel Ave


,sensor_name,sensor_id,flood_start_time,flood_end_time,max_depth_inches,flood_profile_depth_inches,flood_profile_time_secs,duration_mins,geometry,deploy_type,tidally_influenced,in_floodplain,station_id,station_name,dist_to_station_m,lag_hours,high_tide_level_ft
280,SI - Snug Harbor Rd / Kissel Ave,SI-kissel-ave-snug-harbor-rd-2agb9c,2025-08-21 23:12:05+00:00,2025-08-22 00:47:05,2.64,"[0.00, 0.63, 0.67, 0.67, 0.67, 0.75, 1.02, 1.0...","[0, 60, 119, 179, 240, 300, 360, 420, 479, 539...",95.00,POINT (-74.10641 40.64422),pluvial,No,True,8518750,The Battery,9994.396615,-0.582,2.864
393,SI - Snug Harbor Rd / Kissel Ave,SI-kissel-ave-snug-harbor-rd-2agb9c,2025-10-13 05:54:28+00:00,2025-10-13 07:15:28,1.97,"[0.00, 0.43, 0.51, 0.51, 0.67, 0.67, 0.87, 0.9...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",81.00,POINT (-74.10641 40.64422),pluvial,No,True,8518750,The Battery,9994.396615,-0.026,1.731
406,SI - Snug Harbor Rd / Kissel Ave,SI-kissel-ave-snug-harbor-rd-2agb9c,2025-10-13 16:44:26+00:00,2025-10-13 19:17:25,3.15,"[0.00, 0.51, 0.67, 0.71, 0.71, 0.83, 0.83, 0.8...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",152.98,POINT (-74.10641 40.64422),pluvial,No,True,8518750,The Battery,9994.396615,-1.526,2.432
424,SI - Snug Harbor Rd / Kissel Ave,SI-kissel-ave-snug-harbor-rd-2agb9c,2025-10-30 18:09:15+00:00,2025-10-30 20:08:15,7.40,"[0.00, 0.79, 1.02, 1.02, 1.10, 1.10, 1.22, 1.3...","[0, 180, 240, 300, 360, 420, 480, 540, 600, 66...",119.00,POINT (-74.10641 40.64422),pluvial,No,True,8518750,The Battery,9994.396615,-1.046,1.801
607,SI - Snug Harbor Rd / Kissel Ave,SI-kissel-ave-snug-harbor-rd-2agb9c,2026-03-12 03:51:21+00:00,2026-03-12 04:18:21,0.98,"[0.00, 0.87, 0.91, 0.94, 0.94, 0.98, 0.94, 0.9...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",27.01,POINT (-74.10641 40.64422),pluvial,No,True,8518750,The Battery,9994.396615,-3.444,1.090



Q - Beach Chn Dr/ Beach 59th St


,sensor_name,sensor_id,flood_start_time,flood_end_time,max_depth_inches,flood_profile_depth_inches,flood_profile_time_secs,duration_mins,geometry,deploy_type,tidally_influenced,in_floodplain,station_id,station_name,dist_to_station_m,lag_hours,high_tide_level_ft
723,Q - Beach Chn Dr/ Beach 59th St,Q-beach-59th-st-beach-channel-dr-1zbc0d,2026-04-16 23:14:41+00:00,2026-04-17 00:58:44,4.25,"[0.00, 0.63, 0.79, 0.79, 1.02, 1.10, 1.26, 1.4...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",104.06,POINT (-73.78921 40.59408),pluvial,No,True,01311850,JAMAICA BAY AT INWOOD NY,3725.56125,-0.955,4.20
733,Q - Beach Chn Dr/ Beach 59th St,Q-beach-59th-st-beach-channel-dr-1zbc0d,2026-04-17 23:59:36+00:00,2026-04-18 01:44:39,4.72,"[0.00, 0.51, 0.87, 1.06, 1.18, 1.26, 1.26, 1.3...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",105.05,POINT (-73.78921 40.59408),pluvial,No,True,01311850,JAMAICA BAY AT INWOOD NY,3725.56125,-1.007,4.26
760,Q - Beach Chn Dr/ Beach 59th St,Q-beach-59th-st-beach-channel-dr-1zbc0d,2026-04-20 01:35:56+00:00,2026-04-20 03:18:02,4.02,"[0.00, 0.55, 0.75, 0.94, 1.02, 1.34, 1.34, 1.3...","[0, 62, 122, 182, 242, 302, 362, 422, 482, 542...",102.11,POINT (-73.78921 40.59408),pluvial,No,True,01311850,JAMAICA BAY AT INWOOD NY,3725.56125,-1.001,4.11
906,Q - Beach Chn Dr/ Beach 59th St,Q-beach-59th-st-beach-channel-dr-1zbc0d,2026-06-12 21:30:41+00:00,2026-06-12 22:51:45,3.27,"[0.00, 0.00, 0.67, 0.67, 0.83, 0.83, 0.98, 1.3...","[0, 60, 180, 240, 300, 360, 420, 480, 540, 600...",81.05,POINT (-73.78921 40.59408),pluvial,No,True,01311850,JAMAICA BAY AT INWOOD NY,3725.56125,-0.889,4.11
913,Q - Beach Chn Dr/ Beach 59th St,Q-beach-59th-st-beach-channel-dr-1zbc0d,2026-06-13 22:12:30+00:00,2026-06-14 00:17:34,7.44,"[0.00, 0.43, 0.75, 0.83, 1.06, 1.46, 1.50, 1.6...","[0, 60, 120, 180, 240, 300, 360, 420, 480, 540...",125.07,POINT (-73.78921 40.59408),pluvial,No,True,01311850,JAMAICA BAY AT INWOOD NY,3725.56125,-1.192,4.47
936,Q - Beach Chn Dr/ Beach 59th St,Q-beach-59th-st-beach-channel-dr-1zbc0d,2026-06-16 00:07:15+00:00,2026-06-16 02:38:20,11.46,"[0.00, 1.30, 1.61, 1.61, 1.85, 1.85, 2.05, 2.3...","[0, 180, 240, 300, 360, 420, 480, 540, 600, 66...",151.09,POINT (-73.78921 40.59408),pluvial,No,True,01311850,JAMAICA BAY AT INWOOD NY,3725.56125,-1.079,4.81


Upon looking at the data in-depth, both here and on floodnet.nyc, many of these events are repeats. Additionally, many of the sensor graphs show flood depth events in 2025, but do not detect floods as real events in the above dataset. The reason for this is explored below.

### Data-quality check: sensors with zero recorded flood events before 2026

`flood_geo` only contains sensors that already have at least one flood event, so sensors that never logged one are silently absent from every table above. This splits the full `final_sensors_to_show` candidate set into three groups: has events, not yet deployed before 2026 (expected silence), and deployed before 2026 but still silent (the real data-quality question).

In [19]:
# Count pre-2026 flood events per sensor, keyed the same normalized way as the cell 9
# join (strip/lowercase/collapse-whitespace), so this lines up with flood_data exactly.
pre2026_counts = flood_data[flood_data['flood_start_time'] < '2026-01-01'].groupby('_key').size()

# Work from final_sensors_to_show (not flood_geo) because flood_geo only contains sensors
# that already survived the inner join in cell 9 -- a sensor with zero events never makes
# it into flood_geo at all, so it would be invisible if we started from there instead.
sensor_activity = final_sensors_to_show.copy()
sensor_activity['_key'] = sensor_activity['sensor_name'].apply(normalize_name)
sensor_activity['date_deployed'] = pd.to_datetime(sensor_activity['date_deployed'], errors='coerce')
# .map() leaves sensors with no key in pre2026_counts as NaN; fillna(0) means "had events, zero of them"
sensor_activity['pre2026_flood_count'] = sensor_activity['_key'].map(pre2026_counts).fillna(0)

has_events = sensor_activity['pre2026_flood_count'] > 0
not_yet_deployed = sensor_activity['date_deployed'] >= '2026-01-01'

# 'silent_live' = deployed before 2026 but zero logged events -- the real data-quality question.
# np.select evaluates conditions in order and takes the first match; default='unknown' only
# fires if a row matches none of the three conditions, which shouldn't happen since
# has_events and not_yet_deployed are mutually exclusive and jointly exhaustive with silent_live.
sensor_activity['status'] = np.select(
    [has_events, not_yet_deployed, ~has_events & ~not_yet_deployed],
    ['has_events', 'not_yet_deployed', 'silent_live'],
    default='unknown',
)
sensor_activity['status'].value_counts()

status
has_events          75
silent_live         74
not_yet_deployed    51
Name: count, dtype: int64

In [20]:
# Rule out a name-matching bug: how many flood_data sensor names fail to match ANY
# deployed sensor at all (not just the floodplain/tidal candidate set)? This checks the
# join key itself against the full sensor universe, independent of the in_floodplain /
# coastal / tidally_influenced filtering that final_sensors_to_show applies -- so it can't
# be hiding a name-matching bug behind that filter.
all_sensor_keys = set(floodnet_sensor_mapping['sensor_name'].apply(normalize_name))
truly_unmatched = set(flood_data['_key']) - all_sensor_keys
print(f"flood_data sensor names with no match anywhere in final_deployed_sensors: {len(truly_unmatched)}")
sorted(truly_unmatched)

flood_data sensor names with no match anywhere in final_deployed_sensors: 1


['q - 102nd st/160th ave']

In [21]:
# Does silence track sensor health (status) or sensor calibration (mount height above the
# flood threshold)? Compare silent-but-live sensors against sensors that did report floods.
# not_yet_deployed is skipped because its silence is fully explained by deployment timing
# already -- including it here would just add noise to the has_events vs. silent_live comparison.
for status, grp in sensor_activity.groupby('status'):
    if status == 'not_yet_deployed':
        continue  # excluded -- silence there is just deployment timing, not informative
    print(f'--- {status} (n={len(grp)}) ---')
    print(grp['sensor_status'].value_counts())          # device health: good/noisy/dead/etc.
    print(grp['lowest_point_height_delta_inches'].describe())  # calibration: mount height above threshold
    print()

--- has_events (n=75) ---
sensor_status
good               58
dead                6
noisy               5
hardware_issue      2
signal              2
low_charge          1
needs_driverail     1
Name: count, dtype: int64
count    75.000000
mean     10.005867
std       4.795688
min       0.940000
25%       7.560000
50%       9.130000
75%      10.965000
max      29.530000
Name: lowest_point_height_delta_inches, dtype: float64

--- silent_live (n=74) ---
sensor_status
good            59
noisy            5
signal           4
dead             2
non-ota          2
needs_sensor     1
low_charge       1
Name: count, dtype: int64
count    74.000000
mean     11.798919
std       9.357211
min       1.810000
25%       7.800000
50%       9.310000
75%      13.452500
max      62.800000
Name: lowest_point_height_delta_inches, dtype: float64



In [22]:
# Sensor-level detail for reporting: silent-but-live sensors, longest-live first.
# Longest-live-first surfaces the sensors with the most exposure time to have had a flood
# and still didn't -- i.e. the strongest candidates for a genuine data-quality problem.
silent_live_report = sensor_activity.loc[sensor_activity['status'] == 'silent_live', [
    'sensor_name', 'borough', 'date_deployed', 'sensor_status',
    'lowest_point_height_delta_inches', 'in_floodplain', 'tidally_influenced'
]].copy()
silent_live_report['days_live_before_2026'] = (pd.Timestamp('2026-01-01') - silent_live_report['date_deployed']).dt.days
silent_live_report = silent_live_report.sort_values('days_live_before_2026', ascending=False)

silent_live_report.to_csv('silent_sensors_report.csv', index=False)
print(f"{len(silent_live_report)} sensors deployed before 2026 with zero recorded flood events -> silent_sensors_report.csv")
silent_live_report

74 sensors deployed before 2026 with zero recorded flood events -> silent_sensors_report.csv


,sensor_name,borough,date_deployed,sensor_status,lowest_point_height_delta_inches,in_floodplain,tidally_influenced,days_live_before_2026
342,BK - Hoyt St/5th St,Brooklyn,2020-10-05 00:00:00,good,17.17,True,No,1914
401,BX - Sheridan Bl/173rd St,Bronx,2022-02-11 00:00:00,good,62.80,True,No,1420
102,BK - Henry St/Mill St,Brooklyn,2022-02-18 00:00:00,needs_sensor,5.31,True,No,1413
287,BK - Columbia St/Bay St,Brooklyn,2022-02-18 00:00:00,signal,12.83,True,No,1413
114,SI - Jewett Ave/Castleton Ave,Staten Island,2022-07-22 00:00:00,signal,9.25,True,No,1259
...,...,...,...,...,...,...,...,...
146,M - W 29th St/12th Ave,Manhattan,2025-11-07 12:01:00,good,6.61,True,No,54
408,M - W 24th St/12th Ave,Manhattan,2025-11-07 11:57:00,good,8.78,True,No,54
329,M - W 42nd St/11th Ave,Manhattan,2025-11-07 12:09:00,good,4.57,True,No,54
312,BX - Newman Ave/ Gildersleeve Ave,Bronx,2025-11-14 12:00:00,good,8.66,True,No,47


So, of all sensors deployed before 1/1/2026 (the last one before this date being deployed on 11/21/2025), these 74 sensors detected ZERO events in 2025. This is a serious problem because the graphs on floodnet.nyc show clear events in 2025. Why did these sensors not detect them? The next step is to make sure the issue is actually in the dataset, not in my code.

### Verifying the silence against the source directly

The checks above all go through `flood_data`/`flood_geo`, which are built by this notebook's own fetch and join logic. To rule out that logic as the cause of the 2025 silence, this section fetches raw events for the silent sensors straight from `aq7i-eu5q` (the same public NYC Open Data flood-events feed cell 6 reads, queried directly here) in one request -- bypassing `flood_data`, `flood_geo`, and every join in this notebook entirely -- then aggregates by year in pandas. If a sensor still shows a gap here, the gap is in the published dataset, not in anything computed above.

In [23]:
import requests

# SoQL string literals use '' to escape an embedded quote (standard SQL-style escaping) --
# needed because $where below interpolates sensor names directly into the query string.
def soql_escape(s):
    return s.replace("'", "''")

# One request for every silent sensor's full event history (no date filter, no join) --
# batched into a single IN (...) clause instead of one request per sensor, since Socrata
# happily filters server-side and this avoids 74 round trips for the same result.
names_in = ", ".join(f"'{soql_escape(n)}'" for n in silent_live_report['sensor_name'])
r = requests.get('https://data.cityofnewyork.us/resource/aq7i-eu5q.json', params={
    '$select': 'sensor_id, sensor_name, flood_start_time',
    '$where': f'sensor_name IN ({names_in})',
    '$limit': 50000,  # well above the ~1.3k events currently in the whole feed; guards against truncation
})
r.raise_for_status()
raw_events = pd.DataFrame(r.json())
raw_events['yr'] = pd.to_datetime(raw_events['flood_start_time']).dt.year
print(f"{len(raw_events)} historical events found for {raw_events['sensor_name'].nunique()} / {len(silent_live_report)} silent sensors")

120 historical events found for 40 / 74 silent sensors


In [24]:
# One row per sensor, one column per year (all 74 silent sensors, including ones with
# zero events in ANY year -- reindex against silent_live_report's full sensor list so
# those sensors show up as all-NaN rows instead of silently dropping out of the pivot,
# since pivot_table only produces rows for sensors that appear at least once in raw_events).
year_pivot = (
    raw_events.pivot_table(index='sensor_name', columns='yr', values='sensor_id', aggfunc='count')
    .reindex(silent_live_report['sensor_name'])
)
year_pivot.to_csv('silent_sensors_year_breakdown.csv')
year_pivot

yr,2020,2021,2022,2023,2024,2026
sensor_name,,,,,,
BK - Hoyt St/5th St,2.0,4.0,3.0,11.0,6.0,1.0
BX - Sheridan Bl/173rd St,NaN,NaN,NaN,NaN,NaN,NaN
BK - Henry St/Mill St,NaN,NaN,NaN,NaN,NaN,NaN
BK - Columbia St/Bay St,NaN,NaN,NaN,NaN,NaN,NaN
SI - Jewett Ave/Castleton Ave,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
M - W 29th St/12th Ave,NaN,NaN,NaN,NaN,NaN,2.0
M - W 24th St/12th Ave,NaN,NaN,NaN,NaN,NaN,NaN
M - W 42nd St/11th Ave,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
# Flag the strongest evidence of a genuine source-side gap: a sensor with at least one
# event before 2025, none in 2025, and -- critically -- a single sensor_id throughout
# (multiple sensor_ids under one name would mean the name was reused for a different
# physical device, which could fake a "gap" via redeployment instead of an outage).
#
# NOTE: this is the narrowest/strictest cut (requires pre-2025 history). The tiered
# breakdown further down (cells 37-39) supersedes this for reporting purposes -- it
# covers all 74 silent sensors instead of just the ones with pre-2025 history -- but
# is kept here since it's the first place the single_device safety check was added.
single_device = raw_events.groupby('sensor_name')['sensor_id'].nunique().eq(1)

pre2025_years = [y for y in year_pivot.columns if y < 2025]
has_pre2025 = year_pivot[pre2025_years].notna().any(axis=1)
zero_2025 = year_pivot[2025].isna() if 2025 in year_pivot.columns else pd.Series(True, index=year_pivot.index)

confirmed_gap_sensors = year_pivot.index[has_pre2025 & zero_2025 & year_pivot.index.map(single_device).fillna(False)]

print(f"Sensors with a confirmed same-device history before 2025 but a complete 2025 gap: {len(confirmed_gap_sensors)}")
year_pivot.loc[confirmed_gap_sensors].fillna(0).astype(int)

Sensors with a confirmed same-device history before 2025 but a complete 2025 gap: 5


yr,2020,2021,2022,2023,2024,2026
sensor_name,,,,,,
BK - Hoyt St/5th St,2,4,3,11,6,1
Q - Beach 66th St/Thursby Ave,0,0,0,1,0,0
SI - Boundary Ave/ Hull Ave,0,0,0,1,0,0
SI - Olympia Blvd/ Mapleton Ave,0,0,0,1,2,1
BX - Cross St/Minnieford Ave,0,0,0,1,8,0


### How many sensors, really -- ruling out a citywide 2025 outage first

Before trusting sensor-level gaps, rule out the simplest alternative explanation: that FloodNet's whole pipeline had a bad 2025. It didn't -- citywide, `aq7i-eu5q` logged 678 events in 2025 across 151 distinct sensors, more of both than any complete prior year (2024's previous high: 662 events, 53 sensors). 2026 is on pace to exceed it (625 events / 191 sensors already, with the year only ~half elapsed), which is consistent with the network continuing to grow, not with 2025 being some kind of dip. So a sensor that reported nothing all of 2025, while the network overall hit a multi-year high, is a per-sensor anomaly -- not evidence of a platform-wide gap.

The strength of that anomaly depends on how much of 2025 the sensor was actually live for, so `silent_live` is split by exposure: sensors deployed **in** 2025 only had a partial window before 2026 (weak evidence -- they may just not have flooded yet); sensors deployed in 2023-2024 had the *entire* year of 2025 to report and still didn't.

In [26]:
# Citywide baseline: is 2025 low activity network-wide, or just for these sensors?
# Deliberately NOT filtered to the 74 silent sensors -- this is every sensor in the
# public feed, so it answers a different question than everything above: was FloodNet's
# publishing pipeline itself broken for 2025, independent of which sensors we're looking at.
citywide_by_year = requests.get('https://data.cityofnewyork.us/resource/aq7i-eu5q.json', params={
    '$select': 'date_extract_y(flood_start_time) as yr, count(*) as n, count(distinct sensor_name) as sensors',
    '$group': 'yr',
    '$order': 'yr',
}).json()
pd.DataFrame(citywide_by_year)

,yr,n,sensors
0,2020,2,1
1,2021,58,8
2,2022,142,12
3,2023,362,36
4,2024,662,53
5,2025,678,151
6,2026,625,191


In [27]:
# Tier the 74 silent sensors by strength of evidence that 2025 is a genuine gap,
# using date_deployed to separate "live all year and still silent" from "barely live yet".
deploy_dates = silent_live_report.set_index('sensor_name')['date_deployed']

# Same redeployment safeguard as cell 35: if a sensor_name maps to more than one
# sensor_id, its "before" and "after" events could belong to two different physical
# devices, which would fake continuity across the 2025 gap rather than prove it.
device_counts = raw_events.groupby('sensor_name')['sensor_id'].nunique()

def tier(name):
    row = year_pivot.loc[name] if name in year_pivot.index else None
    has_pre = row[pre2025_years].notna().any() if row is not None else False
    has_post = row[2026] if row is not None and 2026 in row.index else float('nan')
    has_post = pd.notna(has_post)
    # A sensor with pre-2025 or 2026 events but MORE than one sensor_id behind that
    # history can't be trusted to prove device continuity -- fall through to the
    # deployment-timing tiers instead of claiming Tier 1/2 on shaky grounds.
    single_device = device_counts.get(name, 0) <= 1
    deployed = deploy_dates.get(name)
    full_year_2025 = pd.notna(deployed) and deployed < pd.Timestamp('2025-01-01')

    if has_pre and has_post and single_device:
        return 'Tier 1: worked before AND after 2025'
    if has_pre and single_device:
        return 'Tier 2: worked before 2025, no 2026 data yet'
    if full_year_2025:
        return 'Tier 3: live all of 2025 (deployed before 2025), zero events until 2026'
    if pd.notna(deployed) and deployed < pd.Timestamp('2026-01-01'):
        return 'Tier 4: deployed during 2025, only a partial window -- weak evidence'
    return 'Never reported any event, any year'

silent_live_report['gap_tier'] = silent_live_report['sensor_name'].apply(tier)
tier_counts = silent_live_report['gap_tier'].value_counts()
print(tier_counts)
print()
# .filter(like=...) matches on the tier label text rather than a fixed list of keys,
# so this stays correct even if a tier label above gets reworded later.
print(f"Tiers 1-3 combined -- sensors with real 2025 exposure and no events (strongest case): "
      f"{tier_counts.filter(like='Tier 1').sum() + tier_counts.filter(like='Tier 2').sum() + tier_counts.filter(like='Tier 3').sum()}")

gap_tier
Tier 3: live all of 2025 (deployed before 2025), zero events until 2026    49
Tier 4: deployed during 2025, only a partial window -- weak evidence       20
Tier 2: worked before 2025, no 2026 data yet                                3
Tier 1: worked before AND after 2025                                        2
Name: count, dtype: int64

Tiers 1-3 combined -- sensors with real 2025 exposure and no events (strongest case): 54


In [28]:
# Full sensor-level detail for the report, ordered by strength of evidence.
# tier_order maps each label to a sort rank since the labels themselves don't sort
# alphabetically in tier order ("Tier 1" < "Tier 2" alphabetically, but "Never..." would
# sort before all of them if left to plain string order).
tier_order = {
    'Tier 1: worked before AND after 2025': 0,
    'Tier 2: worked before 2025, no 2026 data yet': 1,
    'Tier 3: live all of 2025 (deployed before 2025), zero events until 2026': 2,
    'Tier 4: deployed during 2025, only a partial window -- weak evidence': 3,
    'Never reported any event, any year': 4,
}
gap_report = silent_live_report[[
    'sensor_name', 'borough', 'date_deployed', 'sensor_status', 'gap_tier'
]].copy()
gap_report = gap_report.sort_values('gap_tier', key=lambda s: s.map(tier_order))
gap_report.to_csv('silent_sensors_gap_tiers.csv', index=False)
print(f"Wrote silent_sensors_gap_tiers.csv ({len(gap_report)} rows)")
gap_report

Wrote silent_sensors_gap_tiers.csv (74 rows)


,sensor_name,borough,date_deployed,sensor_status,gap_tier
342,BK - Hoyt St/5th St,Brooklyn,2020-10-05 00:00:00,good,Tier 1: worked before AND after 2025
297,SI - Olympia Blvd/ Mapleton Ave,Staten Island,2023-05-19 00:00:00,good,Tier 1: worked before AND after 2025
403,SI - Boundary Ave/ Hull Ave,Staten Island,2023-05-19 00:00:00,good,"Tier 2: worked before 2025, no 2026 data yet"
322,Q - Beach 66th St/Thursby Ave,Queens,2023-03-17 00:00:00,good,"Tier 2: worked before 2025, no 2026 data yet"
184,BX - Cross St/Minnieford Ave,Bronx,2023-10-12 00:00:00,noisy,"Tier 2: worked before 2025, no 2026 data yet"
...,...,...,...,...,...
146,M - W 29th St/12th Ave,Manhattan,2025-11-07 12:01:00,good,"Tier 4: deployed during 2025, only a partial w..."
408,M - W 24th St/12th Ave,Manhattan,2025-11-07 11:57:00,good,"Tier 4: deployed during 2025, only a partial w..."
329,M - W 42nd St/11th Ave,Manhattan,2025-11-07 12:09:00,good,"Tier 4: deployed during 2025, only a partial w..."
312,BX - Newman Ave/ Gildersleeve Ave,Bronx,2025-11-14 12:00:00,good,"Tier 4: deployed during 2025, only a partial w..."


This is a huge problem. 49 sensors were deployed before 2025 and "live" throughout 2025, but don't have any events reported in 2025 (at least on the published dataset). This could be the graph issue I saw manually for certain stations. I will re-run this analysis on the whole FloodNet dataset in a different notebook.

I put "live" in quotes because there's a chance they were dead, or removed and put back, but we don't know.